# ListenBrainz cleaning walkthrough — Andy Chen (DSO 576, Module 6)

This notebook walks through the cleaning in order, using the functions in `clean_listens.py`, so the notebook and the script cannot drift apart.
It reads `listenbrainz_sample.csv` and **writes nothing**. To regenerate `outputs/`, run `uv run python clean_listens.py`.

Rules: `cleaning_rules.md` · Expected record checks: `record_checks_expected.csv`.

**Run:** *Kernel → Restart & Run All* (kernel = this project's `.venv`).

In [1]:
import json, unicodedata
import pandas as pd
import clean_listens as cl

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

## 1. Load the raw file as text and confirm it is unchanged (R1)

In [2]:
raw = cl.load_raw()          # exits if the SHA-256 differs
print(cl.sha256(cl.RAW))
print(raw.shape)
raw.dtypes

463000bf005a3c667cfeb708b44aa12f0d35342e34f467ede3caf924426b816d
(1940, 9)


username          object
listened_at       object
track_name        object
artist_name       object
release_name      object
recording_msid    object
artist_mbids      object
recording_mbid    object
release_mbid      object
dtype: object

## 2. Inspect before cleaning
One row = one listen. Candidate key: `username + listened_at`.

In [3]:
print("exact duplicate rows:          ", raw.duplicated().sum())
print("repeated username+listened_at:  ", raw.duplicated(["username", "listened_at"]).sum())
print("repeated username+recording_msid:", raw.duplicated(["username", "recording_msid"]).sum(), "(replays)")
raw.isna().sum().to_frame("missing")

exact duplicate rows:           0
repeated username+listened_at:   0
repeated username+recording_msid: 610 (replays)


,missing
username,0
listened_at,0
track_name,0
artist_name,0
release_name,754
recording_msid,0
artist_mbids,759
recording_mbid,728
release_mbid,759


In [4]:
raw.groupby("username").agg(rows=("listened_at", "size"),
                             first=("listened_at", "min"),
                             last=("listened_at", "max"),
                             with_mbid=("artist_mbids", lambda s: s.notna().sum()))

,rows,first,last,with_mbid
username,,,,
H-music,500,2019-07-15 16:56:10,2019-11-15 18:03:47,191
John V,226,2026-09-17 01:31:34,2026-09-21 03:51:02,36
iliekcomputers,500,2024-11-16 01:49:07,2024-12-16 08:04:37,388
music_af,214,2025-09-17 13:25:40,2026-03-25 07:32:42,213
s_john_s,500,2019-03-07 13:17:39,2019-03-20 12:58:48,353


## 3. Apply the rules

In [5]:
cl.change_log.clear()
clean = cl.clean(raw)
bridge = cl.build_bridge(clean)
summary = cl.artist_summary(clean)
print(len(raw), "->", len(clean), "rows;", int(clean.is_countable.sum()), "countable")
pd.DataFrame(cl.change_log)

1940 -> 1940 rows; 1882 countable


,rule,action,rows_affected,note
0,R8,removed exact duplicate rows,0,
1,R2,listened_at text -> UTC datetime,1940,0 failed conversions
2,R3,artist_mbids string -> JSON list,1181,0 parse failures
3,R4,flagged rows with a malformed ID,0,
4,R5,track_name: trim + NFC changed text,2,
5,R5,artist_name: trim + NFC changed text,0,
6,R5,release_name: trim + NFC changed text,0,
7,R6,rows whose artist_key merges 2+ raw spellings,280,9 keys
8,R6,' - Topic' suffix removed from artist_key,82,
9,R6,alias table mapped artist_key,33,takanashi yasuharu / -yaiba-


## 4. Types and conversions (R2, R3)

In [6]:
print("timestamp failures:", clean.listened_at_utc.isna().sum())
print("artist_mbids parse failures:", (raw.artist_mbids.notna() & clean.artist_mbids_json.isna()).sum())
clean[["listened_at", "listened_at_utc", "artist_mbids", "artist_mbids_json", "n_credited_artists"]].head(3)

timestamp failures: 0
artist_mbids parse failures: 0


,listened_at,listened_at_utc,artist_mbids,artist_mbids_json,n_credited_artists
0,2024-12-16 08:04:37,2024-12-16 08:04:37+00:00,"['9fff2f8a-21e6-47de-a2b8-7f449929d43f', '73e5e69d-3554-...","[""9fff2f8a-21e6-47de-a2b8-7f449929d43f"", ""73e5e69d-3554-...",2
1,2024-12-16 05:56:01,2024-12-16 05:56:01+00:00,['6bbb3983-ce8a-4971-96e0-7cae73268fc4'],"[""6bbb3983-ce8a-4971-96e0-7cae73268fc4""]",1
2,2024-12-16 05:44:12,2024-12-16 05:44:12+00:00,['eb9eb9f8-c691-4fef-9bdf-0cbffbfa42ee'],"[""eb9eb9f8-c691-4fef-9bdf-0cbffbfa42ee""]",1


## 5. Text before and after (R5, R6)
Display text gets NFC only. The grouping key `artist_key` is casefolded, with the `- Topic` suffix removed and one alias applied. Leading zeros in IDs survive.

In [7]:
changed = clean[clean.track_name != clean.track_name_clean][["track_name", "track_name_clean"]]
changed.assign(before=changed.track_name.map(ascii), after=changed.track_name_clean.map(ascii))

,track_name,track_name_clean,before,after
1514,Wind (ワインド),Wind (ワインド),'Wind (\u30ef\u30a4\u30f3\u30c8\u3099)','Wind (\u30ef\u30a4\u30f3\u30c9)'
1521,Wind (ワインド),Wind (ワインド),'Wind (\u30ef\u30a4\u30f3\u30c8\u3099)','Wind (\u30ef\u30a4\u30f3\u30c9)'


In [8]:
merged = clean.groupby("artist_key")["artist_name"].unique().loc[lambda s: s.map(len) > 1]
merged.map(" | ".join).to_frame("raw spellings merged")

,raw spellings merged
artist_key,
arthrn,Arthrn | arthrn
asian kung-fu generation,ASIAN KUNG-FU GENERATION | Asian Kung-Fu Generation
home made kazoku,HOME MADE Kazoku | Home Made Kazoku
josh maison,Josh Maison | josh maison
mixed matches,Mixed Matches | mixed matches
puffy,Puffy | PUFFY
rogergoon,Rogergoon | rogergoon
rythem,Rythem | RYTHEM
yasuharu takanashi / -yaiba-,Takanashi Yasuharu / -yaiba- | Yasuharu Takanashi / -yaiba-


In [9]:
clean.loc[clean.recording_msid.str.startswith("000"), ["listened_at", "recording_msid"]].head(3)

,listened_at,recording_msid
2,2024-12-16 05:44:12,000c52a6-b412-4a92-a948-ce6fc35bd2e6


## 6. Duplicates and suspected double-scrobbles (R8, R9)
The gap distribution is why the threshold is 60 s: there are almost no repeats between 40 and 60 s.

In [10]:
gaps = clean.seconds_since_prev_same_recording.dropna()
pd.cut(gaps, [0, 10, 20, 30, 40, 60, 90, 120, 300, float("inf")], right=False).value_counts().sort_index()

seconds_since_prev_same_recording
[0.0, 10.0)        11
[10.0, 20.0)        5
[20.0, 30.0)       21
[30.0, 40.0)       19
[40.0, 60.0)        2
[60.0, 90.0)       31
[90.0, 120.0)      61
[120.0, 300.0)     41
[300.0, inf)      419
Name: count, dtype: int64

In [11]:
clean.loc[clean.track_name.eq("Road ii Eternity"),
          ["listened_at", "seconds_since_prev_same_recording", "suspected_double_scrobble"]].head(6)

,listened_at,seconds_since_prev_same_recording,suspected_double_scrobble
667,2019-09-20 14:32:48,1553622.0,False
705,2019-09-02 14:59:06,9.0,True
706,2019-09-02 14:58:57,110.0,False
707,2019-09-02 14:57:07,8.0,True
708,2019-09-02 14:56:59,110.0,False
709,2019-09-02 14:55:09,8.0,True


## 7. Missing data (R10) — kept, not filled

In [12]:
cl.missing_table(raw, clean)

,raw_missing,clean_missing
username,0,0
listened_at,0,0
track_name,0,0
artist_name,0,0
release_name,754,754
recording_msid,0,0
artist_mbids,759,759
recording_mbid,728,728
release_mbid,759,759
listened_at_utc,<NA>,0


In [13]:
clean.groupby("username").has_artist_mbid.mean().round(2).to_frame("MBID coverage")

,MBID coverage
username,
H-music,0.38
John V,0.16
iliekcomputers,0.78
music_af,1.00
s_john_s,0.71


## 8. Suspicious values (R11)

In [14]:
print(clean.listened_at_utc.min(), "to", clean.listened_at_utc.max())
print("flagged times:", clean.flag_suspicious_time.sum(), "| bad IDs:", clean.flag_bad_id.sum(),
      "| placeholder artists:", clean.flag_placeholder_artist.sum())
clean.loc[clean.flag_placeholder_artist, ["username", "listened_at", "track_name", "artist_name", "artist_key"]]

2019-03-07 13:17:39+00:00 to 2026-09-21 03:51:02+00:00
flagged times: 0 | bad IDs: 0 | placeholder artists: 2


,username,listened_at,track_name,artist_name,artist_key
1445,s_john_s,2019-03-10 12:51:10,1. Itachi Uchiha,Unknown,<NA>
1491,s_john_s,2019-03-10 10:26:50,I am Rock Lee!,Unknown,<NA>


## 9. Bridge (listen → credited artist MBID) check

In [15]:
back = bridge.merge(clean[["listen_id"]], on="listen_id", how="left", validate="many_to_one", indicator=True)
print("expected rows:", clean.n_credited_artists.sum(), "| bridge rows:", len(bridge),
      "| unmatched:", (back._merge != "both").sum(), "| rows after merge:", len(back))

expected rows: 1479 | bridge rows: 1479 | unmatched: 0 | rows after merge: 1479


## 10–11. Reconciliation and metric denominators

In [16]:
cl.user_table(raw, clean)

,raw_rows,clean_rows,double_scrobbles,countable_listens,with_artist_mbid,first_listen_utc,last_listen_utc,mbid_coverage,hit_500_cap
username,,,,,,,,,
H-music,500,500,57,443,191,2019-07-15,2019-11-15,38%,True
John V,226,226,0,226,36,2026-09-17,2026-09-21,16%,False
iliekcomputers,500,500,0,500,388,2024-11-16,2024-12-16,78%,True
music_af,214,214,0,214,213,2025-09-17,2026-03-25,100%,False
s_john_s,500,500,1,499,353,2019-03-07,2019-03-20,71%,True


In [17]:
keyed = clean[clean.is_countable & clean.artist_key.notna()]
pairs = keyed.groupby(["artist_key", "username"]).size()
print("countable listens:", clean.is_countable.sum(), "| with artist_key:", len(keyed))
print("listener-artist pairs:", len(pairs), "| returned 2+ days:", summary.returning_listeners.sum(),
      "| return rate:", round(summary.returning_listeners.sum() / len(pairs), 3))
summary.head(10)

countable listens: 1882 | with artist_key: 1880
listener-artist pairs: 751 | returned 2+ days: 180 | return rate: 0.24


,artist_key,countable_listens,distinct_days,display_name,artist_mbid,unique_listeners,returning_listeners,return_rate
61,arthrn,182,19,Arthrn,25d54fa4-0afa-4e47-a1ce-e2cfe5948fda,1,1,1.0
28,alan silvestri,77,1,Alan Silvestri - Topic,<NA>,1,0,0.0
531,roseboy666,75,4,Roseboy666,<NA>,1,1,1.0
726,yasuharu takanashi / -yaiba-,61,3,Takanashi Yasuharu / -yaiba-,<NA>,1,1,1.0
327,kanye west,34,3,Kanye West,164f0d73-1234-4e2c-8743-d77bf2191051,1,1,1.0
422,ministerio3d,25,1,ministerio3d,<NA>,1,0,0.0
676,toshiro masuda,24,3,Toshiro Masuda,bbe94033-40e5-4350-abe6-ac5af4c24079,1,1,1.0
163,diante do trono,16,2,Diante do Trono,<NA>,1,1,1.0
165,diljit dosanjh,16,5,Diljit Dosanjh,f931c961-b647-4861-be8c-f47d84a4de51,1,1,1.0
534,"run the jewels, el-p, killer mike",16,2,"Run The Jewels, El-P, Killer Mike",a80c5dc5-b12e-4667-9f5a-b568961f3839,1,1,1.0


## 12. Record checks (expected values written before the script)

In [18]:
checks = cl.run_record_checks(clean, bridge)
print("all match:", checks["match"].all())
checks[["check_id", "listened_at", "field", "original_value", "expected_value", "actual_value", "match"]]

all match: True


,check_id,listened_at,field,original_value,expected_value,actual_value,match
0,1,2024-12-16 05:44:12,recording_msid,000c52a6-b412-4a92-a948-ce6fc35bd2e6,000c52a6-b412-4a92-a948-ce6fc35bd2e6,000c52a6-b412-4a92-a948-ce6fc35bd2e6,True
1,1,2024-12-16 05:44:12,listened_at_utc,2024-12-16 05:44:12,2024-12-16 05:44:12+00:00,2024-12-16 05:44:12+00:00,True
2,2,2024-12-16 08:04:37,n_credited_artists,"['9fff2f8a-21e6-47de-a2b8-7f449929d43f', '73e5e69d-3554-...",2,2,True
3,2,2024-12-16 08:04:37,bridge_rows,"['9fff2f8a-21e6-47de-a2b8-7f449929d43f', '73e5e69d-3554-...",2,2,True
4,2,2024-12-16 08:04:37,artist_id,"Drake, Rihanna",9fff2f8a-21e6-47de-a2b8-7f449929d43f,9fff2f8a-21e6-47de-a2b8-7f449929d43f,True
5,3,2019-07-15 17:39:39,suspected_double_scrobble,Road ii Eternity (previous same-recording submission 17:...,True,True,True
6,3,2019-07-15 17:39:39,is_countable,Road ii Eternity,False,False,True
7,3,2019-07-15 17:39:08,suspected_double_scrobble,Road ii Eternity (previous same-recording submission 17:...,False,False,True
8,4,2019-11-05 23:02:06,artist_name_clean,arthrn,arthrn,arthrn,True
9,4,2019-11-05 23:02:06,artist_key,arthrn,arthrn,arthrn,True


## 13. Reproducibility
`uv run python check_reproducible.py` reruns the script into a fresh temp folder and compares every output's SHA-256 with `outputs/output_checksums.txt`.

## 14. Limitations
See the *Limitations* section of the report and README: no shared time window across users, 3 users capped at 500 listens, 39% of listens without MBIDs (16%–100% by user), UTC day boundaries, and listening loops (H-music / Roseboy666) that dominate one user's repeat counts.